In [1]:
import pandas as pd
from time import time

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
from feature_engine.outliers import Winsorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score

RT = 42
RANDOM_STATE = 42

import numpy as np
import torch
import torch.nn as nn

RT = 42
RANDOM_STATE = 42
torch.manual_seed(RT)
np.random.seed(RT)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(DEVICE)

mps


In [2]:
# 0. load data

X_test = pd.read_csv("/Users/fangsiyu/Desktop/SDU-2026-fall-code/AML/kaggle/X_test.csv")

X_train = pd.read_csv("/Users/fangsiyu/Desktop/SDU-2026-fall-code/AML/kaggle/X_train.csv")
y_train = pd.read_csv("/Users/fangsiyu/Desktop/SDU-2026-fall-code/AML/kaggle/y_train.csv")
# ==========================================================================================

X_test['Gender'] = X_test['Gender'].map({'Female': 0, 'Male': 1})

X_train['Gender'] = X_train['Gender'].map({'Female': 0, 'Male': 1})
y_train['Diabetes'] = y_train['Diabetes'].map({'No': 0, 'Yes': 1})
# ==========================================================================================

X_train = X_train.set_index('ID')
X_test = X_test.set_index('ID')
y_train = y_train.set_index('ID')['Diabetes']

assert X_train.index.equals(y_train.index)



# 1. train_test_split

X_t, X_val, y_t, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_train
)
print(f"X_t, y_t, {len(X_t), len(y_t)} \nX_val, y_val {len(X_val), len(y_val)}")
print(f"Length of X_test {len(X_test)}")

X_t, y_t, (45066, 45066) 
X_val, y_val (7953, 7953)
Length of X_test 17673


In [3]:
df = pd.concat([X_train, y_train], axis=1).copy()

# 數值欄切成區間
df["BMI_bin"] = pd.cut(df["BMI"], bins=[0, 18.5, 25, 30, 35, 40, 200], right=False,
                       labels=["<18.5", "18.5-25", "25-30", "30-35", "35-40", "40+"])
df["MentHlth_bin"] = pd.cut(df["MentHlth"], bins=[-1, 0, 13, 30], labels=["0", "1-13", "14-30"])
df["PhysHlth_bin"] = pd.cut(df["PhysHlth"], bins=[-1, 0, 13, 30], labels=["0", "1-13", "14-30"])

# 分組用的欄位：原本的欄位，三個數值欄換成區間版
group_cols = [c for c in X_train.columns if c not in ["BMI", "MentHlth", "PhysHlth"]] \
             + ["BMI_bin", "MentHlth_bin", "PhysHlth_bin"]

table = (
    df.groupby(group_cols, observed=True)["Diabetes"]
      .agg(size="size", n_yes="sum", rate="mean")
      .reset_index()
)
table = table[table["size"] >= 2].sort_values("size", ascending=False)
table["mixed"] = (table["rate"] > 0) & (table["rate"] < 1)

n_in = table["size"].sum()
print(f"在人數 >= 2 的組裡: {n_in} / {len(df)} 人 ({n_in / len(df):.1%})")
print(f"組數: {len(table)}，其中答案不一致的組: {table['mixed'].sum()}")

table.to_csv("similar_groups.csv", index=False)
display(table.head(20))

在人數 >= 2 的組裡: 6202 / 53019 人 (11.7%)
組數: 2221，其中答案不一致的組: 792


,HighBP,HighChol,CholCheck,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,...,Education,Income,Gender,BMI_bin,MentHlth_bin,PhysHlth_bin,size,n_yes,rate,mixed
4896,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,38,0,0.000000,False
4957,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,36,0,0.000000,False
5094,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,35,0,0.000000,False
4846,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,30,1,0.033333,True
5033,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,25,0,0.000000,False
5170,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,1,25-30,0,0,18,1,0.055556,True
5156,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,17,0,0.000000,False
5808,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,17,0,0.000000,False
5904,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,0,18.5-25,0,0,16,1,0.062500,True
5824,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,6.0,8.0,1,25-30,0,0,15,1,0.066667,True


In [7]:
from sklearn.inspection import permutation_importance

def add_factors(X):
    X = X.copy()
    X["RiskCount"] = (
        X[["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack", "DiffWalk"]].sum(axis=1)
        + (X["BMI"] >= 30)
    )
    # X["BMI_bin"] = pd.cut(X["BMI"], bins=[0, 18.5, 25, 30, 35, 40, 200], right=False, labels=False)
    # X["MentHlth_bin"] = pd.cut(X["MentHlth"], bins=[-1, 0, 13, 30], labels=False)
    # X["PhysHlth_bin"] = pd.cut(X["PhysHlth"], bins=[-1, 0, 13, 30], labels=False)
    return X

X_t_f = add_factors(X_t)
X_val_f = add_factors(X_val)

hgb = HistGradientBoostingClassifier(
    max_iter=200, max_leaf_nodes=15, min_samples_leaf=50, random_state=RT,
)
cv = StratifiedKFold(5, shuffle=True, random_state=RT)

# 1. CV 比較
s0 = cross_val_score(hgb, X_t, y_t, cv=cv, scoring="accuracy").mean()
s1 = cross_val_score(hgb, X_t_f, y_t, cv=cv, scoring="accuracy").mean()
print(f"HGB 原始 21 欄:     {s0:.4f}")
print(f"HGB + 4 個合併因子: {s1:.4f}\n")

# 2. Permutation importance：在 X_t 上訓練，在 X_val 上算
hgb.fit(X_t_f, y_t)
r = permutation_importance(
    hgb, X_val_f, y_val, scoring="accuracy",
    n_repeats=10, random_state=RT, n_jobs=-1,
)
imp = pd.DataFrame(
    {"mean": r.importances_mean, "std": r.importances_std},
    index=X_val_f.columns,
).sort_values("mean", ascending=False)
display(imp.round(4))


# HGB 原始 21 欄:     0.7550
# HGB + 4 個合併因子: 0.7565

HGB 原始 21 欄:     0.7550
HGB + 4 個合併因子: 0.7565



,mean,std
RiskCount,0.0422,0.0030
GenHlth,0.0415,0.0031
Age,0.0254,0.0020
BMI,0.0171,0.0024
HvyAlcoholConsump,0.0043,0.0006
Gender,0.0034,0.0009
HighBP,0.0033,0.0012
CholCheck,0.0022,0.0006
HighChol,0.0019,0.0007
DiffWalk,0.0016,0.0010


In [8]:
diffs = []
for seed in [0, 1, 2, 3, 4]:
    hgb_s = HistGradientBoostingClassifier(
        max_iter=200, max_leaf_nodes=15, min_samples_leaf=50, random_state=seed,
    )
    cv_s = StratifiedKFold(5, shuffle=True, random_state=seed)

    a = cross_val_score(hgb_s, X_t, y_t, cv=cv_s, scoring="accuracy").mean()
    b = cross_val_score(hgb_s, X_t_f, y_t, cv=cv_s, scoring="accuracy").mean()
    diffs.append(b - a)
    print(f"seed {seed}: 原始 {a:.4f}  +因子 {b:.4f}  差 {b - a:+.4f}")

print(f"\n差距平均 {np.mean(diffs):+.4f}，5 次裡有 {sum(d > 0 for d in diffs)} 次加因子比較高")

seed 0: 原始 0.7551  +因子 0.7557  差 +0.0006
seed 1: 原始 0.7560  +因子 0.7569  差 +0.0010
seed 2: 原始 0.7554  +因子 0.7558  差 +0.0004
seed 3: 原始 0.7551  +因子 0.7563  差 +0.0012
seed 4: 原始 0.7564  +因子 0.7565  差 +0.0001

差距平均 +0.0006，5 次裡有 5 次加因子比較高


In [10]:
BINARY_COLS = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
    'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare',
    'NoDocbcCost', 'DiffWalk', 'Gender',
]
NUMERIC_COLS = ['MentHlth', 'PhysHlth', 'Education', 'Income']
ONEHOT_COLS = ['GenHlth', 'Age']

df["BMI30"] = (df["BMI"] >= 30).astype(int)


rows = []
for c in BINARY_COLS + ["BMI30"]:
    r = df.groupby(c)["Diabetes"].mean()
    rows.append({
        "col": c,
        "rate_0": r.get(0),          # 這欄 = 0 的人，糖尿病比例
        "rate_1": r.get(1),          # 這欄 = 1 的人，糖尿病比例
        "diff": r.get(1) - r.get(0), # 差距：正的代表 1 比較容易有糖尿病
        "share_1": (df[c] == 1).mean(),  # 這欄 = 1 的人佔多少
    })

display(pd.DataFrame(rows).sort_values("diff", ascending=False).round(3))

,col,rate_0,rate_1,diff,share_1
0,HighBP,0.283,0.669,0.386,0.562
2,CholCheck,0.143,0.509,0.366,0.976
12,DiffWalk,0.420,0.739,0.319,0.250
5,HeartDiseaseorAttack,0.456,0.754,0.298,0.148
1,HighChol,0.345,0.640,0.295,0.524
14,BMI30,0.376,0.657,0.280,0.441
4,Stroke,0.484,0.737,0.253,0.063
3,Smoker,0.456,0.549,0.092,0.473
11,NoDocbcCost,0.492,0.576,0.083,0.092
10,AnyHealthcare,0.454,0.502,0.048,0.954


In [11]:
def add_risk(X, extra_risk=(), healthy=False, extra_healthy=()):
    X = X.copy()
    cols = ["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack", "DiffWalk", *extra_risk]
    X["RiskCount"] = X[cols].sum(axis=1) + (X["BMI"] >= 30)
    if healthy:
        h_cols = ["PhysActivity", "Fruits", "Veggies", *extra_healthy]
        X["HealthyCount"] = X[h_cols].sum(axis=1)
    return X

variants = {
    "現在的 RiskCount":  add_risk(X_t),
    "+ CholCheck":       add_risk(X_t, extra_risk=["CholCheck"]),
    "+ HealthyCount":    add_risk(X_t, healthy=True),
    "+ Healthy+Alcohol": add_risk(X_t, healthy=True, extra_healthy=["HvyAlcoholConsump"]),
}

for name, Xv in variants.items():
    scores = []
    for seed in range(5):
        hgb_s = HistGradientBoostingClassifier(
            max_iter=200, max_leaf_nodes=15, min_samples_leaf=50, random_state=seed,
        )
        cv_s = StratifiedKFold(5, shuffle=True, random_state=seed)
        scores.append(cross_val_score(hgb_s, Xv, y_t, cv=cv_s, scoring="accuracy").mean())
    print(f"{name:18s} 平均 {np.mean(scores):.4f}  {[round(s, 4) for s in scores]}")

現在的 RiskCount      平均 0.7562  [np.float64(0.7557), np.float64(0.7569), np.float64(0.7558), np.float64(0.7563), np.float64(0.7565)]
+ CholCheck        平均 0.7563  [np.float64(0.7559), np.float64(0.7565), np.float64(0.7564), np.float64(0.7558), np.float64(0.7569)]
+ HealthyCount     平均 0.7561  [np.float64(0.7557), np.float64(0.7562), np.float64(0.7567), np.float64(0.7555), np.float64(0.7565)]
+ Healthy+Alcohol  平均 0.7558  [np.float64(0.7557), np.float64(0.7556), np.float64(0.7564), np.float64(0.755), np.float64(0.756)]
